# Notebook 3 - RAG Knowledge Base

Builds a retrieval knowledge base from:

- HMS dataset description, feature dictionary, model cards, and XAI explanation cards
- Domain knowledge and ethical limitations
- **PDF literature corpus** in `literature/pdfs/` (student depression, anxiety, help-seeking, etc.)

All documents are chunked, embedded with `all-MiniLM-L6-v2`, and stored for cosine-similarity
retrieval in `outputs/kb/` (no external vector database required).

In [1]:
%pip install -q sentence-transformers pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup and feature dictionary

In [2]:
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = Path("outputs")
KB_DIR = OUTPUT_DIR / "kb"
KB_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_DICT = {
    "age": "Respondent age in years.",
    "gender_male": "Indicator for male gender identity.",
    "gender_female": "Indicator for female gender identity.",
    "international": "Indicator for international student status.",
    "fincur": "Current financial situation / stress (higher = more strain).",
    "finpast": "Financial situation while growing up.",
    "food_worry": "Worry about running out of food (food insecurity).",
    "sleep_wknight": "Hours of sleep on a typical weeknight.",
    "sleep_wkend": "Hours of sleep on a typical weekend night.",
    "exerc": "Frequency of physical exercise.",
    "belong1": "Sense of belonging on campus (item 1).",
    "belong2": "Sense of belonging on campus (item 2).",
    "lonely": "Self-reported loneliness.",
    "aca_impa": "Academic impairment due to emotional/mental health.",
    "flourish": "Flourishing / positive psychological functioning scale.",
    "positiveMH": "Positive mental health indicator.",
    "sub_any": "Any substance use indicator.",
    "binge1pl": "Binge drinking (one or more occasions).",
    "smok_cig": "Cigarette smoking indicator.",
    "discrim_race": "Experiences of racial discrimination.",
    "discrim_gender": "Experiences of gender discrimination.",
    "discrim_sexual": "Experiences of sexual-orientation discrimination.",
    "inst_type": "Institution type.",
    "inst_size": "Institution size.",
    "inst_public": "Whether the institution is public.",
}

## 2. Assemble knowledge base documents

In [3]:
docs = []


def add(doc_id, doc_type, title, text):
    docs.append({"id": doc_id, "type": doc_type, "title": title, "text": text})


add("dataset_overview", "dataset", "Healthy Minds Study 2024-2025 dataset",
    "The Healthy Minds Study (HMS) is a large annual survey of student mental health at US "
    "colleges and universities. This pipeline uses the 2024-2025 wave with institutional "
    "characteristics. The classification target is dep_or_anx: 1 indicates elevated depression "
    "and/or anxiety (screening positive), 0 otherwise. After cleaning (finished surveys, valid "
    "target, at least 60 percent of features present) there are about 63,581 respondents with a "
    "44.9 percent positive rate. Twenty-five features cover demographics, financial stress, sleep "
    "and lifestyle, social support, academics, substance use, discrimination, and institution.")

for f, desc in FEATURE_DICT.items():
    add(f"feature_{f}", "feature", f"Feature: {f}", f"{f}: {desc}")

# performance + explanation cards from notebook 2
for path in sorted(glob.glob(str(OUTPUT_DIR / "cards" / "*.json"))):
    card = json.load(open(path))
    add(f"card_{Path(path).stem}", card.get("card_type", "card"),
        Path(path).stem.replace("_", " "), json.dumps(card, indent=2))

# natural-language summary of aggregated importance (improves retrieval)
agg = json.load(open(OUTPUT_DIR / "cards" / "aggregated_explanation.json"))
summary = "Across all nine models, the most important feature groups for depression-risk classification are: " + \
    "; ".join(f"{g['group']} (importance {g['importance']}, {g['agreement']} agreement)" for g in agg["top_feature_groups"][:5]) + ". " + agg["final_insight"]
add("aggregated_importance_summary", "explanation", "Aggregated feature importance summary", summary)

add("domain_depression_drivers", "domain", "Domain knowledge: correlates of student depression",
    "Research consistently links college student depression and anxiety with loneliness and low "
    "social belonging, academic impairment and stress, financial strain and food insecurity, poor "
    "sleep, low positive mental health/flourishing, substance use, and experiences of discrimination. "
    "These are associations in observational data, not proven causes.")

add("ethics_limitations", "ethics", "Ethical limitations and safe-use guidance",
    "This system is NOT a diagnostic tool and must not replace mental-health professionals. "
    "Predictions are screening signals that may contain bias, especially for sensitive features "
    "such as gender, race, and sexual-orientation discrimination. Sensitive features must be handled "
    "carefully and reported at group level, never used for individual screening without clinical "
    "validation. The system must preserve privacy and de-identification. Any LLM answer must be "
    "grounded only in retrieved evidence and must state uncertainty and model disagreement.")

print(f"Built {len(docs)} KB documents")
pd.Series([d["type"] for d in docs]).value_counts()

Built 43 KB documents


feature                      25
model_performance             9
local_explanation             3
dataset                       1
aggregated_explanation        1
global_feature_importance     1
explanation                   1
domain                        1
ethics                        1
Name: count, dtype: int64

## 2b. Ingest PDF literature corpus

Extract text from every PDF in `literature/pdfs/`, split into overlapping chunks, and add each chunk as a retrievable document tagged `literature`.

In [4]:
import re
from pypdf import PdfReader

PDF_DIR = Path("literature/pdfs")
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150


def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text: str, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()
        if len(piece) >= 80:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


def ingest_pdfs(pdf_dir: Path):
    pdf_paths = sorted(pdf_dir.glob("*.pdf"))
    if not pdf_paths:
        print(f"No PDFs found in {pdf_dir.resolve()}")
        return 0
    n_chunks = 0
    for pdf_path in pdf_paths:
        try:
            reader = PdfReader(str(pdf_path))
            pages = []
            for page in reader.pages:
                pages.append(clean_text(page.extract_text() or ""))
            full_text = clean_text(" ".join(pages))
            chunks = chunk_text(full_text)
            stem = pdf_path.stem
            for i, chunk in enumerate(chunks):
                add(
                    doc_id=f"lit_{stem}_chunk_{i:04d}",
                    doc_type="literature",
                    title=f"{stem} (part {i + 1}/{len(chunks)})",
                    text=chunk,
                )
            n_chunks += len(chunks)
            print(f"  {pdf_path.name}: {len(chunks)} chunks", flush=True)
        except Exception as exc:
            print(f"  SKIP {pdf_path.name}: {exc}", flush=True)
    return n_chunks


print(f"PDF directory: {PDF_DIR.resolve()}")
n_lit = ingest_pdfs(PDF_DIR)
print(f"Added {n_lit} literature chunks from {len(list(PDF_DIR.glob('*.pdf')))} PDFs")
pd.Series([d["type"] for d in docs]).value_counts()

PDF directory: C:\Users\test\Desktop\2007-2025 datasets (.csv, .dta)\xai_rag_depression\literature\pdfs


  2019_gheli_mntlhlthcollege_rp.pdf: 117 chunks


  2024-nsduh-companion-report.pdf: 83 chunks


  2024-State-of-Mental-Health-in-America-Report.pdf: 170 chunks


  A-Student-Primer-on-College-Mental-Health.pdf: 55 chunks


...

  UNCF_MentalHealthReport_Flourishing_FINAL.pdf: 73 chunks


  V34N4.pdf: 69 chunks


  WHPL_Canon_WB_BTtoPWellbeingInitiative.pdf: 78 chunks


Added 2278 literature chunks from 27 PDFs


literature                   2278
feature                        25
model_performance               9
local_explanation               3
dataset                         1
aggregated_explanation          1
global_feature_importance       1
explanation                     1
domain                          1
ethics                          1
Name: count, dtype: int64

## 3. Embed and store

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [f"{d['title']}. {d['text']}" for d in docs]
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=len(texts) > 100).astype(np.float32)
np.save(KB_DIR / "embeddings.npy", emb)
json.dump(docs, open(KB_DIR / "documents.json", "w", encoding="utf-8"), indent=2)

stats = pd.Series([d["type"] for d in docs]).value_counts().to_dict()
stats["total_documents"] = len(docs)
stats["embedding_dim"] = int(emb.shape[1])
json.dump(stats, open(KB_DIR / "kb_stats.json", "w"), indent=2)
print(f"Saved embeddings {emb.shape} and {len(docs)} documents to {KB_DIR}")
print(pd.Series(stats).drop("embedding_dim", errors="ignore"))

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Saved embeddings (2321, 384) and 2321 documents to outputs\kb
literature                   2278
feature                        25
model_performance               9
local_explanation               3
dataset                         1
aggregated_explanation          1
global_feature_importance       1
explanation                     1
domain                          1
ethics                          1
total_documents              2321
dtype: int64


## 4. Retrieval helper and sanity checks

In [6]:
def retrieve(query, k=4):
    q = model.encode([query], normalize_embeddings=True).astype(np.float32)[0]
    sims = emb @ q
    order = np.argsort(sims)[::-1][:k]
    return [(docs[i]["id"], docs[i]["type"], round(float(sims[i]), 3)) for i in order]


for q in ["Why is this student at high depression risk?",
          "What are the most important feature groups?",
          "Is this a clinical diagnosis?",
          "How well do the models perform?",
          "What helps college students with loneliness and depression?",
          "When should a student seek campus counseling?"]:
    print(f"\nQuery: {q}")
    for did, dt, s in retrieve(q):
        print(f"  {s:.3f}  [{dt}] {did}")
print("\nNotebook 3 complete.")


Query: Why is this student at high depression risk?
  0.610  [literature] lit_Mental-Health_Report_chunk_0031
  0.600  [literature] lit_A-Student-Primer-on-College-Mental-Health_chunk_0008
  0.598  [literature] lit_MentalHealth_v6_Interactive_chunk_0101
  0.592  [literature] lit_A-Student-Primer-on-College-Mental-Health_chunk_0023

Query: What are the most important feature groups?
  0.493  [explanation] aggregated_importance_summary
  0.363  [feature] feature_belong1
  0.361  [feature] feature_discrim_race
  0.354  [feature] feature_gender_female

Query: Is this a clinical diagnosis?
  0.392  [literature] lit_Mental_disorders_among_college_students_in_the_WHO_World_Mental_Health_Surve_chunk_0097
  0.391  [literature] lit_Mental_disorders_among_college_students_in_the_WHO_World_Mental_Health_Surve_chunk_0060
  0.389  [literature] lit_Mental_disorders_among_college_students_in_the_WHO_World_Mental_Health_Surve_chunk_0056
  0.386  [literature] lit_Mental_disorders_among_college_students